# Лабораторная работа №17: Diffusion model с нуля и classifier-free guidance

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.

## Цели обучения

После работы студент должен уметь:

- Реализовать forward noising и noise prediction.
- Обучить compact conditional network.
- Добавить CFG и сравнить guidance scales.

## Теоретическая часть

DDPM постепенно добавляет Gaussian noise, затем обучает network обращать процесс. Classifier-free guidance смешивает conditional и unconditional predictions и управляет компромиссом fidelity/diversity.

![DDPM](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/ddpm.png)


DDPM обучает εθ(x_t,t,y) восстанавливать внесённый шум. При classifier-free dropout условие заменяется специальным индексом 10; на inference ε_cfg=ε_u+w(ε_c−ε_u). Для MNIST качество класса проверяет отдельно обученный на train классификатор, его точность на test обязательно сообщается.

Прямой процесс: $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$. CFG: $\hat\epsilon=\epsilon_u+w(\epsilon_c-\epsilon_u)$; при двух вызовах сети на шаг $\mathrm{NFE}=2T$.

### Экспериментальный протокол

Данные MNIST: train первые 20 000, test первые 2 000, без пересечения. Полное обучение: 10 000 шагов; smoke: 2. Основной результат — реальные MNIST checkpoint и условная генерация при разных w. Для продолжения после сбоя сохраняйте artifacts в Drive или задайте CV_ARTIFACTS. Не смешивайте smoke/full checkpoint.

### Материалы

- [DDPM paper](https://arxiv.org/abs/2006.11239)
- [Annotated Diffusion](https://huggingface.co/blog/annotated-diffusion)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.

Версии библиотек, источник данных и лицензия должны быть указаны в отчёте.

## Реализация

`FAST_DEV_RUN=True` проверяет связность пайплайна на малом бюджете и не является итоговым экспериментом. Полная сдача требует `FAST_DEV_RUN=False` и реально полученных метрик.

In [1]:
from pathlib import Path
import os, json, random, time, math, copy
import numpy as np
import torch
SEED = 42
FAST_DEV_RUN = False  # True только для короткой проверки связности, не для сдачи.
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True   # одинаковые числа при повторном прогоне
torch.backends.cudnn.benchmark = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ARTIFACTS = Path(os.environ.get('CV_ARTIFACTS', 'artifacts'))
ARTIFACTS.mkdir(parents=True, exist_ok=True)
RUN_MODE = 'smoke' if FAST_DEV_RUN else 'full'
OUT_DIR = ARTIFACTS / RUN_MODE   # smoke и full чекпоинты/отчёты физически разделены (требование протокола лабы)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT = Path(os.environ.get('CV_COURSE_DATA', str(Path.cwd().parents[1] / 'data')))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
def save_json(name, value):
    destination = OUT_DIR / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')  # UTF-8 обязательно (Windows-локаль cp1251)
import torchvision
_env = {'device': DEVICE, 'fast_dev_run': FAST_DEV_RUN, 'run_mode': RUN_MODE, 'artifacts': str(OUT_DIR),
        'seed': SEED, 'torch': torch.__version__, 'numpy': np.__version__, 'torchvision': torchvision.__version__,
        'data_source': 'torchvision.datasets.MNIST (зеркало ossci-datasets S3)',
        'data_license': 'MNIST - свободно распространяемый датасет (LeCun et al.), используется в учебных целях',
        'model_source': 'все модели лабы (TinyDenoiser, DigitCNN) обучаются с нуля, предобученные веса не используются'}
save_json('lab17_env.json', _env)
print(_env)

{'device': 'cuda', 'fast_dev_run': False, 'run_mode': 'full', 'artifacts': 'artifacts\\full', 'seed': 42, 'torch': '2.14.1+cu126', 'numpy': '2.4.6', 'torchvision': '0.29.1+cu126', 'data_source': 'torchvision.datasets.MNIST (зеркало ossci-datasets S3)', 'data_license': 'MNIST - свободно распространяемый датасет (LeCun et al.), используется в учебных целях', 'model_source': 'все модели лабы (TinyDenoiser, DigitCNN) обучаются с нуля, предобученные веса не используются'}


In [2]:
# Сеть: если HTTPS перехватывается прокси/антивирусом (self-signed сертификат),
# переводим проверку сертификатов на системное хранилище ОС (пакет truststore,
# однократно: pip install truststore). Если пакета нет — стандартная проверка.
try:
    import truststore, urllib3
    truststore.inject_into_ssl()
    urllib3.disable_warnings()
    print('SSL: проверка сертификатов через системное хранилище ОС (truststore)')
except ImportError:
    print('SSL: truststore не установлен — стандартная проверка сертификатов')

SSL: проверка сертификатов через системное хранилище ОС (truststore)


In [3]:
# Проверка зависимостей лабы 17: torch, torchvision (MNIST), numpy, matplotlib.
# diffusers НЕ используется - DDPM реализуется с нуля. MNIST при отсутствии кэша скачается
# автоматически (~65 МБ, ossci-зеркало); в лабах 14/16 он уже скачивался - скорее всего, кэш на месте.
import importlib.util
_need = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'matplotlib': 'matplotlib'}
missing = [k for k in _need if importlib.util.find_spec(k) is None]
if missing:
    raise ImportError('Установите недостающее и перезапустите ячейку:  pip install ' + ' '.join(_need[k] for k in missing))
import torch, torchvision, numpy, matplotlib
_info = {'torch': torch.__version__, 'torchvision': torchvision.__version__,
         'numpy': numpy.__version__, 'matplotlib': matplotlib.__version__,
         'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
if torch.cuda.is_available():
    _p = torch.cuda.get_device_properties(0)
    _info['gpu'] = f'{_p.name} | cc {_p.major}.{_p.minor}'
else:
    _info['gpu_note'] = 'CUDA недоступна - допустимо: модели крошечные, но полный прогон (10000 шагов denoiser) на CPU заметно дольше'
print('Версии:', _info)

Версии: {'torch': '2.14.1+cu126', 'torchvision': '0.29.1+cu126', 'numpy': '2.4.6', 'matplotlib': '3.11.2', 'device': 'cuda', 'gpu': 'NVIDIA GeForce GTX 1050 Ti | cc 6.1'}


### Расписание прямой диффузии

*Вес: 1 балл.*

**Постановка.** Шумовой schedule и форма выхода denoiser определяют корректность всех последующих шагов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Задайте beta, alpha и alpha-bar, проверьте формулу внесения внешнего шума.

**Результат.** Проверяемое состояние «Расписание прямой диффузии» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Задайте beta, alpha и alpha-bar, проверьте формулу внесения внешнего шума» и его результат доступен следующему этапу.

In [4]:
# ЗАДАНИЕ 1 - расписание прямой диффузии: beta, alpha, alpha-bar и проверка формулы q_sample:
# x_t = sqrt(alpha_bar_t)*x_0 + sqrt(1 - alpha_bar_t)*eps  (Ho et al., 2020, DDPM).
T_STEPS = 2 if FAST_DEV_RUN else 500      # число шагов диффузии T (полный режим: 500)
BETA_START, BETA_END = 1e-4, 0.02         # линейный schedule beta_t из статьи DDPM

def diffusion_schedule(t_steps, beta_start=BETA_START, beta_end=BETA_END, device=DEVICE):
    betas = torch.linspace(beta_start, beta_end, t_steps, device=device)   # [T]
    alphas = 1.0 - betas
    abar = torch.cumprod(alphas, dim=0)                                    # alpha_bar_t = prod_{s<=t} alpha_s
    assert bool(torch.all(betas > 0)) and bool(torch.all(betas < 1)), 'beta должна быть в (0,1)'
    assert bool(torch.all(abar[1:] < abar[:-1])) and float(abar[-1]) > 0.0, 'alpha_bar монотонно убывает и остаётся > 0'
    return betas, alphas, abar

BETAS, ALPHAS, ABAR = diffusion_schedule(T_STEPS)

def q_sample(x0, t, noise):
    # внесение внешнего шума на шагах t: [B] long; работает одинаково на CPU и CUDA
    abar_t = ABAR.to(x0.device)[t].view(-1, 1, 1, 1)
    return abar_t.sqrt() * x0 + (1.0 - abar_t).sqrt() * noise

# Проверка 1 - точное совпадение формулы при подстановке конкретного шума (на CPU):
_x0 = torch.rand(8, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 1))
_t = torch.tensor([0, T_STEPS // 4, T_STEPS // 2, 3 * T_STEPS // 4, T_STEPS - 1, 0, 1, T_STEPS - 1])
_eps = torch.randn(8, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 2))
_xt = q_sample(_x0, _t, _eps)
_ab = ABAR.cpu()[_t].view(-1, 1, 1, 1)
assert torch.allclose(_xt, _ab.sqrt() * _x0 + (1.0 - _ab).sqrt() * _eps, atol=1e-6), 'формула q_sample неверна'

# Проверка 2 - статистика прямого процесса: E[x_t] ~= sqrt(abar_t)*E[x0], std ~= sqrt(1 - abar_t)
_ti = T_STEPS // 2
_ab_cpu = ABAR.cpu()
_es = torch.randn(20000, 1, 1, 1, generator=torch.Generator().manual_seed(SEED + 3))
_xts = q_sample(torch.full((20000, 1, 1, 1), 0.5), torch.full((20000,), _ti, dtype=torch.long), _es)
_emp_mean, _emp_std = float(_xts.mean()), float(_xts.std())
_th_mean, _th_std = 0.5 * float(_ab_cpu[_ti].sqrt()), float((1.0 - _ab_cpu[_ti]).sqrt())
assert abs(_emp_mean - _th_mean) < 0.02 and abs(_emp_std - _th_std) < 0.02, 'статистика forward-процесса не сходится с теорией'
save_json('lab17_schedule.json', {'T': T_STEPS, 'beta_start': BETA_START, 'beta_end': BETA_END,
           'alpha_bar_first': float(_ab_cpu[0]), 'alpha_bar_mid': float(_ab_cpu[_ti]),
           'alpha_bar_last': float(_ab_cpu[-1]), 'empirical_mean_t_mid': _emp_mean, 'empirical_std_t_mid': _emp_std,
           'theoretical_mean_t_mid': _th_mean, 'theoretical_std_t_mid': _th_std})
print(f'ЗАДАНИЕ 1 OK: линейный schedule T={T_STEPS} | beta {float(BETAS[0]):.5f}..{float(BETAS[-1]):.5f} | '
      f'alpha_bar: t=0 {float(_ab_cpu[0]):.4f}, t=T//2 {float(_ab_cpu[_ti]):.4f}, t=T-1 {float(_ab_cpu[-1]):.6f}')
print(f'  q_sample при t={_ti}: эмпирически mean {_emp_mean:.4f} / std {_emp_std:.4f} '
      f'против теоретических {_th_mean:.4f} / {_th_std:.4f} | отчёт: lab17_schedule.json')

ЗАДАНИЕ 1 OK: линейный schedule T=500 | beta 0.00010..0.02000 | alpha_bar: t=0 0.9999, t=T//2 0.2779, t=T-1 0.006353
  q_sample при t=250: эмпирически mean 0.2648 / std 0.8506 против теоретических 0.2636 / 0.8498 | отчёт: lab17_schedule.json


### Условный denoiser и проверка форм

*Вес: 1 балл.*

**Постановка.** Шумовой schedule и форма выхода denoiser определяют корректность всех последующих шагов. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Создайте сеть с пространственными блоками и embeddings времени/класса; проверьте форму выхода.

**Результат.** Проверяемое состояние «Условный denoiser и проверка форм» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Создайте сеть с пространственными блоками и embeddings времени/класса; проверьте форму выхода» и его результат доступен следующему этапу.

In [5]:
# ЗАДАНИЕ 2 - условный denoiser: компактный UNet с embeddings времени и класса.
# CFG dropout: класс UNCOND_IDX=10 означает "без условия"; сеть ОДНА - условная и безусловная
# ветви отличаются только подставленной меткой (поэтому их можно смешивать при обучении и разделять при CFG).
import torch.nn as nn
import torch.nn.functional as F

UNCOND_IDX = 10            # специальный индекс безусловной ветви (по протоколу лабы)
N_CLASSES_ALL = 11         # 10 цифр (0..9) + безусловный класс
TIME_DIM, EMB_DIM = 64, 128

def timestep_embedding(t, dim=TIME_DIM):
    # синусоидальное кодирование timestep (как в Transformer); t: [B] long
    half = dim // 2
    freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)           # [B, dim]

class ResBlock(nn.Module):
    # residual-блок с инъекцией embedding времени/класса (аддитивная модуляция по каналам)
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(EMB_DIM, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class TinyDenoiser(nn.Module):
    # eps_theta(x_t, t, y): [B,1,28,28] -> [B,1,28,28]; энкодер 28->14->7, декодер со skip-связями
    def __init__(self, base=32, n_classes=N_CLASSES_ALL):
        super().__init__()
        self.cls_emb = nn.Embedding(n_classes, EMB_DIM)
        self.time_mlp = nn.Sequential(nn.Linear(TIME_DIM, EMB_DIM), nn.SiLU(), nn.Linear(EMB_DIM, EMB_DIM))
        self.stem = nn.Conv2d(1, base, 3, padding=1)
        self.inc = ResBlock(base, base)                                        # 28x28
        self.down1 = nn.Conv2d(base, base, 4, stride=2, padding=1)             # 28 -> 14
        self.mid1 = ResBlock(base, base * 2)                                   # 14x14
        self.down2 = nn.Conv2d(base * 2, base * 2, 4, stride=2, padding=1)     # 14 -> 7
        self.mid2 = ResBlock(base * 2, base * 2)                               # 7x7 - бутылочное горлышко
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(base * 2, base * 2, 3, padding=1))  # 7 -> 14
        self.dec2 = ResBlock(base * 4, base)                                   # skip от mid1
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='nearest'),
                                 nn.Conv2d(base, base, 3, padding=1))          # 14 -> 28
        self.dec1 = ResBlock(base * 2, base)                                   # skip от inc
        self.out_norm = nn.GroupNorm(8, base)
        self.out_conv = nn.Conv2d(base, 1, 3, padding=1)

    def forward(self, x_t, t, y):
        emb = self.time_mlp(timestep_embedding(t)) + self.cls_emb(y)
        h1 = self.inc(F.silu(self.stem(x_t)), emb)
        h2 = self.mid1(self.down1(h1), emb)
        h3 = self.mid2(self.down2(h2), emb)
        u = self.dec2(torch.cat([self.up2(h3), h2], dim=1), emb)
        u = self.dec1(torch.cat([self.up1(u), h1], dim=1), emb)
        return self.out_conv(F.silu(self.out_norm(u)))

DENOISER = TinyDenoiser().to(DEVICE)
_x = torch.rand(4, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 4)).to(DEVICE)
_t = torch.tensor([0, T_STEPS // 3, 2 * T_STEPS // 3, T_STEPS - 1], device=DEVICE)
_y = torch.tensor([0, 3, 9, UNCOND_IDX], device=DEVICE)
_eps_pred = DENOISER(_x, _t, _y)
assert _eps_pred.shape == _x.shape, 'форма выхода denoiser обязана совпадать с формой x_t'
DENOISER.zero_grad(); _eps_pred.mean().backward()
assert DENOISER.stem.weight.grad is not None and DENOISER.cls_emb.weight.grad is not None, 'градиенты не доходят до стема/эмбеддинга класса'
DENOISER.eval()
with torch.no_grad():
    _diff = float((DENOISER(_x, _t, torch.zeros_like(_y)) - DENOISER(_x, _t, torch.ones_like(_y))).abs().max())
assert _diff > 1e-5, 'выход не зависит от класса - условная ветка не подключена'
N_DENOISER_PARAMS = sum(p.numel() for p in DENOISER.parameters())
print(f'ЗАДАНИЕ 2 OK: TinyDenoiser {N_DENOISER_PARAMS} параметров | вход {tuple(_x.shape)} -> выход {tuple(_eps_pred.shape)} | '
      f'эмбеддинги t/класса подключены (max |eps(x,y=0)-eps(x,y=1)| {_diff:.4f})')

ЗАДАНИЕ 2 OK: TinyDenoiser 415041 параметров | вход (4, 1, 28, 28) -> выход (4, 1, 28, 28) | эмбеддинги t/класса подключены (max |eps(x,y=0)-eps(x,y=1)| 1.1164)


### Train/test MNIST для DDPM

*Вес: 1 балл.*

**Постановка.** CFG работает только если условная и безусловная ветки видели данные на обучении. Smoke не заменяет обучение полного режима. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Определите фиксированный непересекающийся split и DataLoader.

**Результат.** Проверяемое состояние «Train/test MNIST для DDPM» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Определите фиксированный непересекающийся split и DataLoader» и его результат доступен следующему этапу.

In [6]:
# ЗАДАНИЕ 3 - MNIST для DDPM: train = ПЕРВЫЕ 20000, test = ПЕРВЫЕ 2000 (фиксированный префикс
# датасета; train/test MNIST - разные исходные выборки, пересечение исключено по построению).
# Поток батчей - детерминированная предвычисленная последовательность индексов с фиксированным
# seed (аналог DataLoader с фиксированным generator): одинаковая при любом перезапуске/resume.
import torchvision

MNIST_DIR = DATA_ROOT / 'mnist'
_mnist_train_raw = torchvision.datasets.MNIST(str(MNIST_DIR), train=True, download=True)
_mnist_test_raw = torchvision.datasets.MNIST(str(MNIST_DIR), train=False, download=True)

TRAIN_N = 200 if FAST_DEV_RUN else 20000   # протокол: train первые 20 000
TEST_N = 200 if FAST_DEV_RUN else 2000     # протокол: test первые 2 000
TRAIN_X = _mnist_train_raw.data[:TRAIN_N].float().unsqueeze(1) / 255.0   # [N,1,28,28] в [0,1]
TRAIN_Y = _mnist_train_raw.targets[:TRAIN_N].long()
TEST_X = _mnist_test_raw.data[:TEST_N].float().unsqueeze(1) / 255.0
TEST_Y = _mnist_test_raw.targets[:TEST_N].long()
assert TRAIN_X.size(0) == TRAIN_N and TEST_X.size(0) == TEST_N
assert float(TRAIN_X.min()) >= 0.0 and float(TRAIN_X.max()) <= 1.0
assert not torch.equal(TRAIN_X[:64], TEST_X[:64]), 'train и test не должны пересекаться'
print('данные: train', tuple(TRAIN_X.shape), '/ test', tuple(TEST_X.shape),
      '| диапазон [0,1]:', round(float(TRAIN_X.min()), 1), '-', round(float(TRAIN_X.max()), 1),
      '| классы train:', TRAIN_Y.bincount(minlength=10).tolist(),
      '| классы test:', TEST_Y.bincount(minlength=10).tolist())

def ddpm_batches(seed, n, steps, batch):
    # фиксированный поток батчей: одна и та же последовательность индексов при любом перезапуске
    g = torch.Generator().manual_seed(seed)
    return [torch.randint(0, n, (batch,), generator=g) for _ in range(steps)]

данные: train (20000, 1, 28, 28) / test (2000, 1, 28, 28) | диапазон [0,1]: 0.0 - 1.0 | классы train: [1994, 2281, 1929, 2076, 1945, 1775, 1971, 2093, 1922, 2014] | классы test: [175, 234, 219, 207, 217, 179, 178, 205, 192, 194]


### Обучение epsilon-предсказателя с CFG dropout

*Вес: 1 балл.*

**Постановка.** CFG работает только если условная и безусловная ветки видели данные на обучении. Smoke не заменяет обучение полного режима. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите условную и безусловную ветви, сохраняя модель и состояние оптимизатора.

**Результат.** Проверяемое состояние «Обучение epsilon-предсказателя с CFG dropout» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите условную и безусловную ветви, сохраняя модель и состояние оптимизатора» и его результат доступен следующему этапу.

In [7]:
# ЗАДАНИЕ 4 - обучение epsilon-предсказателя с classifier-free dropout: в p=10% случаев батча
# условие заменяется на UNCOND_IDX, поэтому ОДНА сеть учит и условную, и безусловную ветви.
# Loss = MSE(eps_pred, eps). Чекпоинт + resume + EMA; smoke и full не смешиваются (разные каталоги).
CFG_DROP_P = 0.1
DDPM_STEPS = 2 if FAST_DEV_RUN else 10000   # протокол: полное обучение 10 000 шагов
DDPM_BATCH = 64 if FAST_DEV_RUN else 128
DDPM_LR = 2e-4

class EMA:
    # экспоненциальное скользящее среднее весов; сэмплирование в DDPM стандартно ведётся из EMA-копии
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1.0 - self.decay)
            else:
                self.shadow[k].copy_(v)

_d_opt = torch.optim.Adam(DENOISER.parameters(), lr=DDPM_LR)
_d_ckpt = OUT_DIR / 'lab17_ckpt_denoiser.pt'
_d_batches = ddpm_batches(SEED + 20, TRAIN_N, DDPM_STEPS, DDPM_BATCH)
_ema = EMA(DENOISER)
_d_hist, _start = [], 0
if _d_ckpt.exists():
    _ck = torch.load(_d_ckpt, map_location=DEVICE, weights_only=False)
    DENOISER.load_state_dict(_ck['model']); _d_opt.load_state_dict(_ck['optimizer'])
    _ema.shadow = {k: v.to(DEVICE) for k, v in _ck['ema'].items()}
    _start, _d_hist = _ck['step'], _ck['loss_history']
    torch.set_rng_state(_ck['rng']['cpu'].cpu())                       # восстановление RNG -> точное продолжение
    if _ck['rng'].get('cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in _ck['rng']['cuda']])
    print('denoiser: resume с шага', _start)

DENOISER.train()
_t0 = time.time()
_log_every = max(1, DDPM_STEPS // 10)
for _step in range(_start, DDPM_STEPS):
    _x0 = TRAIN_X[_d_batches[_step]].to(DEVICE)
    _y = TRAIN_Y[_d_batches[_step]].clone().to(DEVICE)
    _y[torch.rand(_y.size(0), device=DEVICE) < CFG_DROP_P] = UNCOND_IDX   # classifier-free dropout
    _t = torch.randint(0, T_STEPS, (_x0.size(0),), device=DEVICE)
    _eps = torch.randn(_x0.shape, device=DEVICE)
    _x_t = q_sample(_x0, _t, _eps)
    _loss = F.mse_loss(DENOISER(_x_t, _t, _y), _eps)
    _d_opt.zero_grad(); _loss.backward(); _d_opt.step()
    _ema.update(DENOISER)
    _d_hist.append(float(_loss.item()))
    if (_step + 1) % _log_every == 0 or _step + 1 == DDPM_STEPS:
        _recent = _d_hist[-50:]
        print(f'  denoiser шаг {_step + 1}/{DDPM_STEPS} | MSE avg50 {sum(_recent) / len(_recent):.4f} | {time.time() - _t0:.0f} c')
DDPM_TRAIN_TIME = time.time() - _t0
_rng = {'cpu': torch.get_rng_state()}
if torch.cuda.is_available(): _rng['cuda'] = torch.cuda.get_rng_state_all()
torch.save({'model': DENOISER.state_dict(), 'optimizer': _d_opt.state_dict(), 'ema': _ema.shadow,
            'step': DDPM_STEPS, 'loss_history': _d_hist, 'steps': DDPM_STEPS, 'lr': DDPM_LR,
            'cfg_drop_p': CFG_DROP_P, 'T': T_STEPS, 'rng': _rng}, _d_ckpt)
save_json('lab17_denoiser_training.json', {'steps': DDPM_STEPS, 'batch': DDPM_BATCH, 'lr': DDPM_LR, 'T': T_STEPS,
           'cfg_drop_p': CFG_DROP_P, 'loss_first': _d_hist[0], 'loss_last': _d_hist[-1],
           'ema_decay': 0.999, 'train_time_s': round(DDPM_TRAIN_TIME, 1), 'history_every_50': _d_hist[::50]})
print(f'ЗАДАНИЕ 4 OK: MSE {_d_hist[0]:.4f} -> {_d_hist[-1]:.4f} за {DDPM_STEPS} шагов '
      f'(T={T_STEPS}, CFG dropout p={CFG_DROP_P}, EMA 0.999), {DDPM_TRAIN_TIME:.1f} c | чекпоинт: {_d_ckpt.name}')

  denoiser шаг 1000/10000 | MSE avg50 0.0316 | 106 c
  denoiser шаг 2000/10000 | MSE avg50 0.0261 | 212 c
  denoiser шаг 3000/10000 | MSE avg50 0.0235 | 318 c
  denoiser шаг 4000/10000 | MSE avg50 0.0236 | 425 c
  denoiser шаг 5000/10000 | MSE avg50 0.0233 | 533 c
  denoiser шаг 6000/10000 | MSE avg50 0.0225 | 640 c
  denoiser шаг 7000/10000 | MSE avg50 0.0221 | 748 c
  denoiser шаг 8000/10000 | MSE avg50 0.0210 | 855 c
  denoiser шаг 9000/10000 | MSE avg50 0.0219 | 963 c
  denoiser шаг 10000/10000 | MSE avg50 0.0214 | 1070 c
ЗАДАНИЕ 4 OK: MSE 1.0835 -> 0.0209 за 10000 шагов (T=500, CFG dropout p=0.1, EMA 0.999), 1069.7 c | чекпоинт: lab17_ckpt_denoiser.pt


### Независимый классификатор для оценки

*Вес: 1 балл.*

**Постановка.** CFG работает только если условная и безусловная ветки видели данные на обучении. Smoke не заменяет обучение полного режима. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Обучите только на train и измерьте точность классификатора на real test.

**Результат.** Проверяемое состояние «Независимый классификатор для оценки» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Обучите только на train и измерьте точность классификатора на real test» и его результат доступен следующему этапу.

In [8]:
# ЗАДАНИЕ 5 - независимый проверяющий классификатор DigitCNN (как в лабе 16): обучение ТОЛЬКО
# на train (первые 20000), точность на независимом real test (первые 2000). Не связан ни с
# denoiser, ни с его условием - нужен для измерения class agreement генераций.
class DigitCNN(nn.Module):
    # Вход: [B,1,28,28] в [0,1] - тот же диапазон, что у реального MNIST и декодированных сэмплов.
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),    # 28 -> 14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2))   # 14 -> 7
        self.classifier = nn.Linear(32 * 7 * 7, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

CLF = DigitCNN().to(DEVICE)
_z = CLF(torch.rand(2, 1, 28, 28, generator=torch.Generator().manual_seed(SEED + 4)).to(DEVICE))
assert _z.shape == (2, 10)
CLF_PARAMS = sum(p.numel() for p in CLF.parameters())
CLS_STEPS = 2 if FAST_DEV_RUN else 1000
CLS_BATCH = 256
_clf_opt = torch.optim.Adam(CLF.parameters(), lr=1e-3)
_clf_ckpt = OUT_DIR / 'lab17_ckpt_classifier.pt'
_clf_batches = ddpm_batches(SEED + 40, TRAIN_N, CLS_STEPS, CLS_BATCH)
_start = 0
if _clf_ckpt.exists():
    _ck = torch.load(_clf_ckpt, map_location=DEVICE, weights_only=False)
    CLF.load_state_dict(_ck['model']); _clf_opt.load_state_dict(_ck['optimizer']); _start = _ck['step']
    print('классификатор: resume с шага', _start)
CLF.train()
_t0 = time.time()
for _step in range(_start, CLS_STEPS):
    _idx = _clf_batches[_step]
    _loss = F.cross_entropy(CLF(TRAIN_X[_idx].to(DEVICE)), TRAIN_Y[_idx].to(DEVICE))
    _clf_opt.zero_grad(); _loss.backward(); _clf_opt.step()
CLS_TIME = time.time() - _t0

@torch.no_grad()
def clf_eval(x_all, y_all):
    # точность и средняя уверенность на независимом real test + разрез по классам
    CLF.eval()
    correct = torch.zeros(10, dtype=torch.long); total = torch.zeros(10, dtype=torch.long)
    conf_sum, n = 0.0, 0
    for i in range(0, x_all.size(0), 512):
        x, y = x_all[i:i + 512].to(DEVICE), y_all[i:i + 512].to(DEVICE)
        probs = F.softmax(CLF(x), dim=1)
        pred = probs.argmax(1)
        conf_sum += float(probs.max(1).values.sum().item()); n += y.numel()
        for c in range(10):
            m = y == c
            total[c] += int(m.sum()); correct[c] += int((pred[m] == c).sum())
    return float(correct.sum().item() / total.sum().item()), correct.tolist(), total.tolist(), conf_sum / n

CLF_ACC, CLF_CORRECT, CLF_TOTAL, CLF_CONF = clf_eval(TEST_X, TEST_Y)
torch.save({'model': CLF.state_dict(), 'optimizer': _clf_opt.state_dict(), 'step': CLS_STEPS}, _clf_ckpt)
save_json('lab17_classifier_report.json', {'steps': CLS_STEPS, 'batch': CLS_BATCH, 'params': CLF_PARAMS,
           'test_accuracy': CLF_ACC, 'mean_confidence': CLF_CONF,
           'per_class_correct': CLF_CORRECT, 'per_class_total': CLF_TOTAL, 'train_time_s': round(CLS_TIME, 1)})
print(f'ЗАДАНИЕ 5 OK: DigitCNN {CLF_PARAMS} параметров | точность на независимом real test {CLF_ACC:.4f} '
      f'({int(sum(CLF_CORRECT))}/{int(sum(CLF_TOTAL))}), средняя уверенность {CLF_CONF:.4f}, {CLS_TIME:.1f} c')

ЗАДАНИЕ 5 OK: DigitCNN 20490 параметров | точность на независимом real test 0.9770 (1954/2000), средняя уверенность 0.9740, 8.7 c


### Сбалансированные классы и общий шум

*Вес: 1 балл.*

**Постановка.** Сравнение guidance требует одинаковых начального шума и генератора для каждой шкалы. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Создайте сбалансированные классы и фиксированный исходный шум для всех весов CFG.

**Результат.** Проверяемое состояние «Сбалансированные классы и общий шум» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Создайте сбалансированные классы и фиксированный исходный шум для всех весов CFG» и его результат доступен следующему этапу.

In [9]:
# ЗАДАНИЕ 6 - сбалансированные классы и ОБЩИЙ исходный шум для всех весов CFG:
# один и тот же x_T и одна и та же последовательность пошагового шума z у каждой шкалы -
# разница между прогонами определяется только guidance w, а не удачей шума.
GEN_PER_CLASS = 1 if FAST_DEV_RUN else 8
GEN_LABELS = torch.arange(10).repeat_interleave(GEN_PER_CLASS)      # сбалансированно: 0,0,...,9,9
GEN_N = GEN_LABELS.size(0)
assert torch.bincount(GEN_LABELS, minlength=10).min().item() == GEN_PER_CLASS, 'классы должны быть сбалансированы'
X_T_SEED = SEED + 50
SAMPLING_SEED = SEED + 51     # при каждом прогоне генератор пошагового шума z переинициализируется этим seed
_noise_g = torch.Generator().manual_seed(X_T_SEED)
X_T_INIT = torch.randn(GEN_N, 1, 28, 28, generator=_noise_g).to(DEVICE)   # общий стартовый шум x_T для всех w

def make_gen(seed):
    # детерминированный генератор на устройстве семплирования (CUDA/CPU)
    return torch.Generator(device=DEVICE).manual_seed(seed) if DEVICE == 'cuda' else torch.Generator().manual_seed(seed)

print(f'ЗАДАНИЕ 6 OK: {GEN_N} изображений (по {GEN_PER_CLASS} на каждый класс 0..9) | '
      f'общий x_T {tuple(X_T_INIT.shape)} зафиксирован (seed {X_T_SEED}) | пошаговый шум z: seed {SAMPLING_SEED} у всех весов CFG')

ЗАДАНИЕ 6 OK: 80 изображений (по 8 на каждый класс 0..9) | общий x_T (80, 1, 28, 28) зафиксирован (seed 92) | пошаговый шум z: seed 93 у всех весов CFG


### DDPM sampling и guidance

*Вес: 1 балл.*

**Постановка.** Сравнение guidance требует одинаковых начального шума и генератора для каждой шкалы. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сэмплируйте обученной сетью при разных CFG; учитывайте conditional/unconditional NFE.

**Результат.** Проверяемое состояние «DDPM sampling и guidance» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сэмплируйте обученной сетью при разных CFG; учитывайте conditional/unconditional NFE» и его результат доступен следующему этапу.

In [10]:
# ЗАДАНИЕ 7 - DDPM sampling с classifier-free guidance: eps_hat = eps_u + w*(eps_c - eps_u).
# Учёт NFE: w=0 -> 1 вызов сети на шаг, w>0 -> 2 вызова (unconditional + conditional);
# для T шагов это T или 2T вызовов на прогон.
W_SCALES = [0.0, 1.0, 3.0, 5.0]     # шкалы guidance зафиксированы ДО просмотра результатов

@torch.no_grad()
def guided_eps(model, x_t, t_scalar, y, w):
    # безусловная ветка вызывается всегда; условная добавляется только при w>0
    t = torch.full((x_t.size(0),), t_scalar, device=x_t.device, dtype=torch.long)
    eps_u = model(x_t, t, torch.full_like(y, UNCOND_IDX))
    if w == 0:
        return eps_u, 1
    eps_c = model(x_t, t, y)
    return eps_u + w * (eps_c - eps_u), 2

@torch.no_grad()
def ddpm_sample(model, y, w, x_T, gen_seed, stochastic=True):
    # обратный процесс: t = T-1 ... 0;
    # x_{t-1} = 1/sqrt(a_t) * (x_t - (1-a_t)/sqrt(1-abar_t) * eps) + sqrt(beta_t) * z, z ~ N(0,1) при t>0
    # stochastic=False -> детерминированный DDIM-вариант (eta=0) - используется в бонусном задании
    model.eval()
    x = x_T.clone()
    g = make_gen(gen_seed)
    nfe = 0
    for t in range(T_STEPS - 1, -1, -1):
        eps, calls = guided_eps(model, x, t, y, w)
        nfe += calls
        mean = (x - (1.0 - ALPHAS[t]) / (1.0 - ABAR[t]).sqrt() * eps) / ALPHAS[t].sqrt()
        if t > 0:
            if stochastic:
                z = torch.randn(x.shape, generator=g, device=x.device)
                x = mean + BETAS[t].sqrt() * z
            else:
                x0_hat = (x - (1.0 - ABAR[t]).sqrt() * eps) / ABAR[t].sqrt()
                x = ABAR[t - 1].sqrt() * x0_hat.clamp(0.0, 1.0) + (1.0 - ABAR[t - 1]).sqrt() * eps
        else:
            x = mean
    return x.clamp(0.0, 1.0), nfe

SAMPLE_MODEL = copy.deepcopy(DENOISER)          # сэмплируем из EMA-весов (стандарт DDPM)
SAMPLE_MODEL.load_state_dict(_ema.shadow)
SAMPLE_MODEL.eval().to(DEVICE)

# связность: при фиксированном seed повторный прогон обязан дать те же изображения
_img_a, _ = ddpm_sample(SAMPLE_MODEL, GEN_LABELS.to(DEVICE), 0.0, X_T_INIT, SAMPLING_SEED)
_img_b, _ = ddpm_sample(SAMPLE_MODEL, GEN_LABELS.to(DEVICE), 0.0, X_T_INIT, SAMPLING_SEED)
assert torch.equal(_img_a, _img_b), 'семплирование должно быть детерминированным при фиксированном seed'

GEN_IMAGES, GEN_RESULTS = {}, {}
for _w in W_SCALES:
    _t0 = time.time()
    _imgs, _nfe = ddpm_sample(SAMPLE_MODEL, GEN_LABELS.to(DEVICE), _w, X_T_INIT, SAMPLING_SEED)
    GEN_IMAGES[_w] = _imgs.cpu()
    GEN_RESULTS[_w] = {'nfe': _nfe, 'nfe_per_step': 1 if _w == 0 else 2, 'time_s': round(time.time() - _t0, 1)}
    print(f'  w={_w}: NFE {_nfe} ({GEN_RESULTS[_w]["nfe_per_step"]} вызов(а)/шаг x {T_STEPS} шагов) | {GEN_RESULTS[_w]["time_s"]} c')
print(f'ЗАДАНИЕ 7 OK: генерация при w={W_SCALES} завершена | суммарный NFE {sum(r["nfe"] for r in GEN_RESULTS.values())} '
      f'(conditional/unconditional вызовы учтены)')

  w=0.0: NFE 500 (1 вызов(а)/шаг x 500 шагов) | 9.1 c
  w=1.0: NFE 1000 (2 вызов(а)/шаг x 500 шагов) | 18.1 c
  w=3.0: NFE 1000 (2 вызов(а)/шаг x 500 шагов) | 17.9 c
  w=5.0: NFE 1000 (2 вызов(а)/шаг x 500 шагов) | 17.9 c
ЗАДАНИЕ 7 OK: генерация при w=[0.0, 1.0, 3.0, 5.0] завершена | суммарный NFE 3500 (conditional/unconditional вызовы учтены)


### Сохранение измерений генерации

*Вес: 1 балл.*

**Постановка.** Сравнение guidance требует одинаковых начального шума и генератора для каждой шкалы. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Сохраните метрики, изображения и точность проверяющего классификатора на test.

**Результат.** Проверяемое состояние «Сохранение измерений генерации» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Сохраните метрики, изображения и точность проверяющего классификатора на test» и его результат доступен следующему этапу.

In [11]:
# ЗАДАНИЕ 8 - измерения генерации: class agreement у независимого классификатора, уверенность,
# разнообразие (L2 до ближайшего соседа среди сэмплов) и близость к real-test; PNG-сетки и отчёт.
import matplotlib.pyplot as plt

@torch.no_grad()
def clf_probs(x01):
    CLF.eval()
    return F.softmax(CLF(x01.to(DEVICE)), dim=1).cpu()

def gen_metrics(imgs):
    # imgs: [N,1,28,28] в [0,1]; GEN_LABELS - целевые классы
    probs = clf_probs(imgs)
    agreement = float((probs.argmax(1) == GEN_LABELS).float().mean())
    conf = float(probs.max(1).values.mean())
    flat = imgs.reshape(imgs.size(0), -1)
    d_in = torch.cdist(flat, flat); d_in.fill_diagonal_(float('inf'))
    diversity = float(d_in.min(1).values.mean())                     # средняя L2 до ближайшего сэмпла (ниже -> коллапс)
    d_test = torch.cdist(flat, TEST_X.reshape(TEST_X.size(0), -1))
    to_test = float(d_test.min(1).values.mean())                     # средняя L2 до ближайшего real-test изображения
    return {'agreement': agreement, 'confidence': conf, 'diversity_l2': diversity, 'nearest_test_l2': to_test}

_flat_t = TEST_X.reshape(TEST_X.size(0), -1)
_dd = torch.cdist(_flat_t, _flat_t); _dd.fill_diagonal_(float('inf'))
REF_L2 = float(_dd.min(1).values.mean())     # ориентир: real-test до ближайшего другого real-test
GEN_METRICS = {w: gen_metrics(imgs) for w, imgs in GEN_IMAGES.items()}

def save_grid(imgs, path, per_class):
    # строки = классы 0..9, столбцы = сэмплы класса
    f, ax = plt.subplots(10, per_class, figsize=(max(1.2 * per_class, 2.5), 12.5),
                         squeeze=False, constrained_layout=True)
    for c in range(10):
        for j in range(per_class):
            ax[c, j].imshow(imgs[c * per_class + j, 0], cmap='gray', vmin=0, vmax=1)
            ax[c, j].axis('off')
        ax[c, 0].text(-0.08, 0.5, str(c), transform=ax[c, 0].transAxes, ha='right', va='center', fontsize=8)
    f.savefig(path, dpi=110)
    plt.close(f)

for _w, _imgs in GEN_IMAGES.items():
    save_grid(_imgs, OUT_DIR / f'lab17_gen_w{int(_w)}.png', GEN_PER_CLASS)

save_json('lab17_generation_report.json',
          {'reference_nearest_test_l2': round(REF_L2, 3), 'classifier_test_accuracy': CLF_ACC,
           'n_images': GEN_N, 'per_class': GEN_PER_CLASS, 'x_T_seed': X_T_SEED, 'sampling_seed': SAMPLING_SEED,
           'per_w': {str(w): {**{k: round(v, 4) for k, v in m.items()}, **GEN_RESULTS[w]} for w, m in GEN_METRICS.items()}})
print(f'ориентир: средняя L2 real-test до ближайшего другого real-test {REF_L2:.3f} '
      f'(сэмплы сопоставимы => не дальше от manifold, чем реальные цифры друг от друга)')
for _w, _m in GEN_METRICS.items():
    print(f'  w={_w}: agreement {_m["agreement"]:.3f} | уверенность {_m["confidence"]:.3f} | '
          f'разнообразие L2 {_m["diversity_l2"]:.3f} | до test {_m["nearest_test_l2"]:.3f} | NFE {GEN_RESULTS[_w]["nfe"]}')
_best = max((_m['agreement'] for _w, _m in GEN_METRICS.items() if _w > 0), default=0.0)
if not FAST_DEV_RUN and _best <= GEN_METRICS[0.0]['agreement']:
    print('ВНИМАНИЕ: ни одна guided шкала не превзошла w=0 по agreement - проверьте обучение denoiser')
print('PNG:', ', '.join(f'lab17_gen_w{int(w)}.png' for w in GEN_IMAGES), '| отчёт: lab17_generation_report.json')
print(f'ЗАДАНИЕ 8 OK: метрики/изображения сохранены | точность проверяющего классификатора на test {CLF_ACC:.4f}')

ориентир: средняя L2 real-test до ближайшего другого real-test 5.202 (сэмплы сопоставимы => не дальше от manifold, чем реальные цифры друг от друга)
  w=0.0: agreement 0.125 | уверенность 0.844 | разнообразие L2 7.144 | до test 6.185 | NFE 500
  w=1.0: agreement 0.950 | уверенность 0.969 | разнообразие L2 6.994 | до test 5.623 | NFE 1000
  w=3.0: agreement 0.988 | уверенность 0.991 | разнообразие L2 6.137 | до test 5.502 | NFE 1000
  w=5.0: agreement 1.000 | уверенность 1.000 | разнообразие L2 5.671 | до test 5.766 | NFE 1000
PNG: lab17_gen_w0.png, lab17_gen_w1.png, lab17_gen_w3.png, lab17_gen_w5.png | отчёт: lab17_generation_report.json
ЗАДАНИЕ 8 OK: метрики/изображения сохранены | точность проверяющего классификатора на test 0.9770


### Шумовая ошибка на разных временах

*Вес: 1 балл.*

**Постановка.** Выбор T и schedule влияет на величину остаточного сигнала, а метрика MSE по отложенным изображениям проверяет предсказатель непосредственно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Измерьте conditional/unconditional MSE по фиксированному test batch на нескольких timesteps.

**Результат.** Проверяемое состояние «Шумовая ошибка на разных временах» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Измерьте conditional/unconditional MSE по фиксированному test batch на нескольких timesteps» и его результат доступен следующему этапу.

In [12]:
# ЗАДАНИЕ 9 - шумовая ошибка denoiser на фиксированном test batch при нескольких timestep'ах:
# MSE(eps_pred, eps) для условной ветви (реальные метки) и безусловной (класс 10);
# предсказатель - EMA-модель, eps и batch фиксированы seed'ом.
EVAL_N = min(256, TEST_N)
_x0e = TEST_X[:EVAL_N].to(DEVICE)
_ye = TEST_Y[:EVAL_N].to(DEVICE)
_epse = torch.randn(_x0e.shape, generator=torch.Generator().manual_seed(SEED + 60)).to(DEVICE)
TS_TO_EVAL = sorted(set([0, T_STEPS // 10, T_STEPS // 4, T_STEPS // 2, 3 * T_STEPS // 4, T_STEPS - 1]))
MSE_TABLE = {}
SAMPLE_MODEL.eval()
for _ti in TS_TO_EVAL:
    _tb = torch.full((EVAL_N,), _ti, dtype=torch.long, device=DEVICE)
    _xt = q_sample(_x0e, _tb, _epse)
    with torch.no_grad():
        _pred_c = SAMPLE_MODEL(_xt, _tb, _ye)
        _pred_u = SAMPLE_MODEL(_xt, _tb, torch.full_like(_ye, UNCOND_IDX))
    _mc, _mu = float(F.mse_loss(_pred_c, _epse)), float(F.mse_loss(_pred_u, _epse))
    MSE_TABLE[_ti] = {'mse_cond': _mc, 'mse_uncond': _mu, 'gap_uncond_minus_cond': _mu - _mc}
    print(f'  t={_ti:>4}: MSE cond {_mc:.4f} | uncond {_mu:.4f} | gap {_mu - _mc:+.4f}')
save_json('lab17_noise_mse.json', {'T': T_STEPS, 'eval_images': EVAL_N, 'timesteps': TS_TO_EVAL,
           'table': {str(k): {kk: round(vv, 6) for kk, vv in v.items()} for k, v in MSE_TABLE.items()},
           'note': 'условие помогает при малых/средних t (ниже MSE условной ветви); при t -> T обе стремятся к 1: x_t почти чистый шум'})
print('ЗАДАНИЕ 9 OK: шумовая ошибка измерена на timesteps', TS_TO_EVAL,
      f'| фиксированный test batch {EVAL_N} изображений, seed eps {SEED + 60} | отчёт: lab17_noise_mse.json')

  t=   0: MSE cond 0.1704 | uncond 0.1716 | gap +0.0011
  t=  50: MSE cond 0.0471 | uncond 0.0478 | gap +0.0007
  t= 125: MSE cond 0.0262 | uncond 0.0283 | gap +0.0021
  t= 250: MSE cond 0.0121 | uncond 0.0156 | gap +0.0035
  t= 375: MSE cond 0.0030 | uncond 0.0037 | gap +0.0007
  t= 499: MSE cond 0.0006 | uncond 0.0006 | gap +0.0001
ЗАДАНИЕ 9 OK: шумовая ошибка измерена на timesteps [0, 50, 125, 250, 375, 499] | фиксированный test batch 256 изображений, seed eps 102 | отчёт: lab17_noise_mse.json


### Худшие ошибки denoiser

*Вес: 1 балл.*

**Постановка.** Выбор T и schedule влияет на величину остаточного сигнала, а метрика MSE по отложенным изображениям проверяет предсказатель непосредственно. Отдельная проверка этого шага не позволяет скрыть ошибку за итоговым показателем.

Выполните следующие действия:

1. Зафиксируйте реальные ошибки и пять худших test изображений.

**Результат.** Проверяемое состояние «Худшие ошибки denoiser» и названные в действии файлы при их создании.

**Критерий принятия.** Задание считается принятым, если выполнено действие «Зафиксируйте реальные ошибки и пять худших test изображений» и его результат доступен следующему этапу.

In [13]:
# ЗАДАНИЕ 10 - реальные ошибки denoiser: по-изображениям MSE(eps_pred, eps) на фиксированном
# t = T//2 по отложенному test batch; пять худших изображений с визуальной диагностикой.
import matplotlib.pyplot as plt

FIXED_T_ERR = T_STEPS // 2
_tb = torch.full((EVAL_N,), FIXED_T_ERR, dtype=torch.long, device=DEVICE)
_xt = q_sample(_x0e, _tb, _epse)
with torch.no_grad():
    _pc = SAMPLE_MODEL(_xt, _tb, _ye)
_per_img = ((_pc - _epse) ** 2).mean(dim=(1, 2, 3))          # MSE на каждое изображение
_top = torch.topk(_per_img, 5).indices                        # 5 худших

_f, _ax = plt.subplots(5, 4, figsize=(9, 11), constrained_layout=True)
for _c, _tt in enumerate(['x0 (оригинал)', f'x_t (t={FIXED_T_ERR})', 'eps (истинный)', 'eps_pred (EMA)']):
    _ax[0, _c].set_title(_tt, fontsize=9)
for _r, _k in enumerate(_top.tolist()):
    _ax[_r, 0].imshow(_x0e[_k, 0].cpu(), cmap='gray', vmin=0, vmax=1)
    _ax[_r, 1].imshow(_xt[_k, 0].cpu(), cmap='gray')         # без фикс. vmin/vmax: большой динамический диапазон
    _ax[_r, 2].imshow(_epse[_k, 0].cpu(), cmap='gray', vmin=-3, vmax=3)
    _ax[_r, 3].imshow(_pc[_k, 0].cpu(), cmap='gray', vmin=-3, vmax=3)
    _ax[_r, 0].text(-0.12, 0.5, f'MSE {_per_img[_k].item():.3f}\nкласс {_ye[_k].item()}',
                    transform=_ax[_r, 0].transAxes, ha='right', va='center', fontsize=8)
    for _c in range(4):
        _ax[_r, _c].axis('off')
_f.savefig(OUT_DIR / 'lab17_worst_denoiser_errors.png', dpi=110)
plt.close(_f)
save_json('lab17_worst_errors.json', {'t': FIXED_T_ERR, 'eval_images': EVAL_N,
           'mean_per_image_mse': round(float(_per_img.mean()), 6),
           'top5_image_idx': _top.tolist(),
           'top5_mse': [round(float(_per_img[_k]), 4) for _k in _top],
           'top5_true_labels': [int(_ye[_k].item()) for _k in _top]})
print(f'ЗАДАНИЕ 10 OK: 5 худших ошибок при t={FIXED_T_ERR} (средняя по-изображениям MSE {float(_per_img.mean()):.4f}) | '
      f'PNG: lab17_worst_denoiser_errors.png | отчёт: lab17_worst_errors.json')

ЗАДАНИЕ 10 OK: 5 худших ошибок при t=250 (средняя по-изображениям MSE 0.0121) | PNG: lab17_worst_denoiser_errors.png | отчёт: lab17_worst_errors.json


### Творческое мини-исследование

*Вес: 1 балл, бонус сверх десяти обязательных заданий.*

**Постановка.** Контролируемая абляция позволяет отделить влияние одного фактора от шума выборки. Гипотеза формулируется до просмотра отложенных результатов.

Выполните следующие действия:

1. Сформулируйте собственную предметную гипотезу о поведении модели и измените один фактор, сохраняя те же реальные входы, seed и вычислительный бюджет.
2. Сравните с основным экспериментом, не используя test для выбора настройки; объясните наблюдаемые расхождения.

**Результат.** Отдельные код, измеренная таблица и интерпретация в отчёте.

**Критерий принятия.** Задание считается принятым, если гипотеза проверена на реальных входах, сохранены фактические измерения и показана контролируемая разница. Бонус не является обязательным для Run All.

In [14]:
# БОНУС (1 балл сверх десяти обязательных) - творческое мини-исследование.
# Гипотеза (сформулирована ДО просмотра результатов): детерминированный обратный процесс
# (DDIM, eta=0) при том же w=3 и том же общем x_T сохранит class agreement близким к
# стохастическому DDPM, но СНИЗИТ разнообразие сэмплов, т.к. убирает стохастический
# источник вариативности между сэмплами.
# Контроль: изменён ровно ОДИН фактор (стохастичность реверс-процесса); модель (EMA), x_T,
# метки, seed пошагового шума и порядок NFE фиксированы. Test для выбора настройки не используется.
_imgs_ddim, _nfe_ddim = ddpm_sample(SAMPLE_MODEL, GEN_LABELS.to(DEVICE), 3.0, X_T_INIT, SAMPLING_SEED, stochastic=False)
_m_s, _m_d = GEN_METRICS[3.0], gen_metrics(_imgs_ddim.cpu())
save_json('lab17_bonus_stochasticity.json',
          {'hypothesis': 'DDIM (eta=0) при w=3 сохраняет class agreement, но снижает разнообразие сэмплов',
           'fixed': {'w': 3.0, 'model': 'EMA denoiser', 'x_T_seed': X_T_SEED, 'z_seed': SAMPLING_SEED, 'T': T_STEPS},
           'factor': 'стохастичность реверс-процесса: DDPM (sqrt(beta_t)*z) против DDIM (eta=0)',
           'ddpm_w3': {k: round(v, 4) for k, v in _m_s.items()},
           'ddim_w3': {k: round(v, 4) for k, v in _m_d.items()},
           'nfe_ddim': _nfe_ddim})
print(f'БОНУС OK: w=3 | DDPM (стохаст.): agreement {_m_s["agreement"]:.3f}, разнообразие {_m_s["diversity_l2"]:.3f} | '
      f'DDIM (eta=0): agreement {_m_d["agreement"]:.3f}, разнообразие {_m_d["diversity_l2"]:.3f} | '
      f'NFE {_nfe_ddim} | отчёт: lab17_bonus_stochasticity.json')
# Интерпретация для отчёта: сравнить agreement/diversity двух режимов и объяснить расхождение с гипотезой.

БОНУС OK: w=3 | DDPM (стохаст.): agreement 0.988, разнообразие 6.137 | DDIM (eta=0): agreement 0.000, разнообразие 8.540 | NFE 1000 | отчёт: lab17_bonus_stochasticity.json


## Анализ результатов

Сопоставьте конфигурации при одинаковых данных, seeds и бюджете. Приведите таблицу фактических метрик и времени, не менее пяти неудачных примеров с объяснением причин, ограничения и проверяемые выводы. Не подменяйте измерения утверждением об успешном запуске.

## Требования к сдаче

Перед сдачей перезапустите ядро и выполните все ячейки по порядку в полном режиме. Приложите конфигурацию, версии библиотек, источники/лицензии данных и модели, checkpoints, таблицу измерений, изображения и краткий анализ ошибок. Для длительного обучения перенесите каталог `artifacts/` на Google Drive и возобновляйте обучение с последнего checkpoint; отсутствие GPU или данных нельзя заменять синтетическими метриками.

## Итоги (cv17-summary)

**Протокол.** Устройство: cuda — NVIDIA GeForce GTX 1050 Ti (cc 6.1). Версии: torch 2.14.1+cu126, torchvision 0.29.1+cu126, numpy 2.4.6, matplotlib 3.11.2. Режим: полный (FAST_DEV_RUN=False), seed 42; smoke и full физически разделены (`artifacts/full`), чекпоинты с resume, RNG-состояние сохраняется и восстанавливается.

**Данные (задание 3).** MNIST train 20 000 / test 2 000 — фиксированные префиксы исходных выборок torchvision (зеркало ossci-datasets S3), пересечение исключено по построению; диапазон [0,1]. Распределение классов: train 1775–2281, test 175–234 изображений на цифру. Поток батчей — детерминированная последовательность индексов с фиксированным seed, одинаковая при любом перезапуске. Модели (TinyDenoiser, DigitCNN) обучены с нуля, предобученные веса не используются.

**Задание 1 (расписание прямой диффузии).** Линейный schedule T=500, beta 0.0001 → 0.02; alpha_bar монотонно убывает: 0.9999 (t=0) → 0.2779 (t=250) → 0.0064 (t=499). Формула q_sample проверена дважды: точное совпадение при подстановке конкретного шума и статистика процесса при t=250 — эмпирически mean 0.2648 / std 0.8506 против теоретических 0.2636 / 0.8498.

**Задание 2 (условный denoiser).** TinyDenoiser 415 041 параметр: энкодер 28→14→7 на ResBlock'ах с skip-связями, синусоидальный embedding времени + embedding класса (UNCOND_IDX=10, одна сеть на обе ветви CFG). Форма выхода совпадает с входом; условная ветка активна: max |ε(x,y=0)−ε(x,y=1)| = 1.1164; градиенты доходят до стема и эмбеддинга класса.

**Задание 4 (обучение ε-предсказателя).** 10 000 шагов, батч 128, lr 2e-4, CFG dropout p=0.1, EMA 0.999: MSE 1.0835 → 0.0209 (avg50 в конце ≈ 0.021), время 1069.7 c. Сэмплирование ведётся из EMA-копии весов; чекпоинт `lab17_ckpt_denoiser.pt`.

**Задание 5 (проверяющий классификатор).** DigitCNN 20 490 параметров, независим от denoiser и его условия, обучение только на train — 8.7 c. Точность на независимом real test **0.9770** (1954/2000), средняя уверенность 0.9740 — метрику class agreement можно считать достоверной.

**Задания 6–8 (сбалансированная генерация с CFG).** 80 изображений (по 8 на каждый класс 0..9); у всех шкал общий x_T (seed 92) и общий пошаговый шум (seed 93) — разница между прогонами только в w. Ориентир близости: средняя L2 real-test до ближайшего соседа 5.202.

| w | agreement | уверенность | разнообразие L2 | L2 до test | NFE | время |
|---|---|---|---|---|---|---|
| 0.0 | 0.125 | 0.844 | 7.144 | 6.185 | 500 | 9.1 c |
| 1.0 | 0.950 | 0.969 | 6.994 | 5.623 | 1000 | 18.1 c |
| 3.0 | 0.988 | 0.991 | 6.137 | 5.502 | 1000 | 17.9 c |
| 5.0 | 1.000 | 1.000 | 5.671 | 5.766 | 1000 | 17.9 c |

**Задания 9–10 (диагностика denoiser).** MSE(ε_pred, ε) на фиксированном test batch (256 изображений, seed eps 102):

| t | cond | uncond | gap |
|---|---|---|---|
| 0 | 0.1704 | 0.1716 | +0.0011 |
| 50 | 0.0471 | 0.0478 | +0.0007 |
| 125 | 0.0262 | 0.0283 | +0.0021 |
| 250 | 0.0121 | 0.0156 | +0.0035 |
| 375 | 0.0030 | 0.0037 | +0.0007 |
| 499 | 0.0006 | 0.0006 | +0.0001 |

MSE монотонно падает с ростом t — при больших t сигнал почти уничтожен, предсказывать шум легко. Условная ветка точнее безусловной на всех t; разрыв максимален в среднем диапазоне (t=250: +0.0035) — именно там условие полезнее всего. Пять худших по-изображениям ошибок при t=250 (средняя MSE 0.0121) сохранены в `lab17_worst_denoiser_errors.png`.

**Бонус (DDPM против DDIM, eta=0).** Гипотеза сформулирована до просмотра результатов: «DDIM при том же w=3 и том же общем x_T сохранит class agreement близким к DDPM, но снизит разнообразие». Изменён ровно один фактор — стохастичность реверс-процесса; модель (EMA), x_T, метки, seed шума и NFE (1000) фиксированы. Результат: DDPM — agreement 0.988, разнообразие 6.137; DDIM — agreement **0.000**, разнообразие **8.540**. Гипотеза **опровергнута по обоим пунктам**: детерминированная траектория при T=500 и чекпоинте на 10 000 шагов уходит с manifold (классификатор не узнаёт ни одной цифры), а стохастический шум работает стабилизатором обратного процесса, а не только источником вариативности.

**Время этапов.** Denoiser 1069.7 c; классификатор 8.7 c; четыре генерации 63.0 c (суммарный NFE 3500); диагностика и бонус отдельно не хронометрировались. Артефакты в `artifacts/full/`: отчёты `lab17_env / schedule / denoiser_training / classifier_report / generation_report / noise_mse / worst_errors / bonus_stochasticity` (.json), чекпоинты `lab17_ckpt_denoiser.pt`, `lab17_ckpt_classifier.pt`, сетки `lab17_gen_w{0,1,3,5}.png`, `lab17_worst_denoiser_errors.png`.

**Вывод.** Classifier-free guidance монотонно управляет компромиссом fidelity/diversity: без guidance модель почти не следует целевым классам (agreement 0.125), при w=1–5 agreement растёт до 1.000 и уверенность до 1.000, а разнообразие сжимается (7.144 → 5.671). Рабочая точка — w=3: agreement 0.988 при большем разнообразии, чем у w=5, и минимальной L2 до real test (5.502). Каждая цифра таблиц — измерение с фиксированными seed'ами, воспроизводимое перезапуском ядра, а не утверждение об успешном запуске.